# Journey into Sound  

Here we will look into the applications of deeplearning in the domain of audio. We will learn how to work with torchaudio library.


## ESC-50 dataset  

We will work with ESC-50 dataset which is a popular audio dataset with labeled collection of 2000 environmental audio recordings organized in 50 classes.  

You can access the dataset using this link : [click here](https://github.com/karolpiczak/ESC-50)  

Let's have a look at our dataset first.

In [1]:
import glob
from collections import Counter

esc50_list = [f.split('-')[-1].replace('wav',"")
              for f in glob.glob("./datasets/ESC-50/audio/*.wav")
              ]

Counter(esc50_list)

Counter()

In [2]:
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset    

class ESC50(Dataset):

    def __init__(self, path):
        files=Path(path).glob('*.wav')
        self.items = [(f,int(f.name.split('-')[-1].replace('.wav','')))
                      for f in files]
        
        self.length = len(self.items)

    def __getitem__(self, index):
        filename, label = self.items[index]
        audio_tensor, sample_rate = torchaudio.load(filename) 
        return audio_tensor,label
    
    def __len__(self):
        return self.length

In [3]:
test_esc50 = ESC50('./datasets/ESC-50/audio/test')
train_esc50 = ESC50('./datasets/ESC-50/audio/train')
valid_esc50 = ESC50('./datasets/ESC-50/audio/valid')



In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_esc50, batch_size=8, shuffle=True)
test_loader = DataLoader(test_esc50, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_esc50, batch_size=8, shuffle=True)



Since we are done with loading the dataset. Let's design out Neural Network for the classification

In [5]:
import torch.nn as nn

class AudioNet(nn.Module):
    def __init__(self):
        super(AudioNet,self).__init__()
        self.conv1 = nn.Conv1d(1, 128, 5, 4)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(7)
        self.conv2 = nn.Conv1d(128, 128, 3)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(7)
        self.conv3 = nn.Conv1d(128, 256, 3)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool3 = nn.MaxPool1d(6)
        self.conv4 = nn.Conv1d(256, 512, 3)
        self.bn4 = nn.BatchNorm1d(512)
        self.pool4 = nn.MaxPool1d(6)
        self.avgpool = nn.AvgPool1d(30)
        self.fc1 = nn.Linear(512,50)

    def forward(self,x):
        x = self.conv1(x)
        x = nn.functional.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = nn.functional.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = nn.functional.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = nn.functional.relu(self.bn4(x))
        x = self.pool4(x)
        x = self.avgpool(x)
        x = x.permute(0,2,1)
        x = self.fc1(x)
        return nn.functional.log_softmax(x, dim=2).squeeze(1)

Now let's initialize our optimizer and loss function and get started.  
We will use Adam optimizer as usual and Cross Entropy loss for the loss function as this is a multi-class classification task.

In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
import torchinfo

audio_net = AudioNet()

torchinfo.summary(audio_net)

Layer (type:depth-idx)                   Param #
AudioNet                                 --
├─Conv1d: 1-1                            768
├─BatchNorm1d: 1-2                       256
├─MaxPool1d: 1-3                         --
├─Conv1d: 1-4                            49,280
├─BatchNorm1d: 1-5                       256
├─MaxPool1d: 1-6                         --
├─Conv1d: 1-7                            98,560
├─BatchNorm1d: 1-8                       512
├─MaxPool1d: 1-9                         --
├─Conv1d: 1-10                           393,728
├─BatchNorm1d: 1-11                      1,024
├─MaxPool1d: 1-12                        --
├─AvgPool1d: 1-13                        --
├─Linear: 1-14                           25,650
Total params: 570,034
Trainable params: 570,034
Non-trainable params: 0

In [ ]:
import torch.optim as optimizer

audio_net.to(device)
torch.save(audio_net.state_dict(),'models/audionet.pth')

loss_fn = nn.CrossEntropyLoss()
optim = optimizer.Adam(audio_net.parameters(), lr=0.01)

Let's use the find_lr function from our learning_rate notebook to find figure out a better lr.

In [ ]:
import math

def find_lr(model, loss_fn, optimizer, init_val=1e-8, final_val=1):
    number_in_epoch = len(train_loader)-1
    update_step = (final_val / init_val)**(1/number_in_epoch)
    lr = init_val
    optimizer.param_groups[0]["lr"] = lr
    best_loss = 0.0
    batch_num = 0
    losses = []
    log_lrs = []

    for data in train_loader:
        batch_num+=1
        inputs,labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        print(f"Labels shape: {labels.shape}, Labels dtype: {labels.dtype}")
        
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs,labels)

        if batch_num > 1 and loss > 4*best_loss:
            return log_lrs[10:-5], losses[10:-5]

        # Record best loss

        if loss < best_loss or batch_num == 1:
            best_loss = loss

        losses.append(loss)
        log_lrs.append(math.log10(lr))

        lr*=update_step
        optimizer.param_groups[0]['lr'] = lr
    
    return log_lrs[10:-5], losses[10:-5]

In [ ]:
import matplotlib.pyplot as plt


log,losses = find_lr(audio_net,loss_fn,optim)
found_lr = 1e-2
plt.plot(log,losses)